# 4. ARIMA next-day prediction

Predict **entire next day** using **ARIMA**: for each (hour, frequency, threshold, class), use the last **window** days of au_pct as input, fit ARIMA, and forecast the next day's au_pct.

- **Error:** same as notebook 2 — computed over the entire day → **one MAE and one RMSE per day**; same testing data and error reporting structure.
- **Final visualization:** dropdown for **class**; show testing MAE/RMSE per day. **Final results table:** MAE (ARIMA) per class.

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
from tqdm.auto import tqdm

from statsmodels.tsa.arima.model import ARIMA

In [12]:
# Path to final data (same as notebook 2)
work_dir = None
for candidate in [Path("organized"), Path("../organized"), Path("work_dir"), Path("../work_dir")]:
    fd = candidate / "final"
    if fd.exists():
        work_dir = candidate
        break
if work_dir is None:
    raise FileNotFoundError('Final dir not found. Tried: organized/final, ../organized/final, work_dir/final, ../work_dir/final')
final_dir = work_dir / "final"
training_dir = final_dir / "training"
testing_dir = final_dir / "testing"

In [13]:
def load_final_parquet(split_dir: Path, band: str, date_yyyymmdd: str) -> Optional[pd.DataFrame]:
    """Load final_<date>.parquet for a given split and band. Returns None if missing."""
    path = split_dir / band / f"final_{date_yyyymmdd}.parquet"
    if not path.exists():
        return None
    return pd.read_parquet(path)


def get_dates_and_bands(split_dir: Path):
    """Return sorted list of date_yyyymmdd and list of unique bands."""
    dates = set()
    bands = []
    for band_dir in sorted(split_dir.iterdir()):
        if not band_dir.is_dir():
            continue
        if band_dir.name not in bands:
            bands.append(band_dir.name)
        for p in band_dir.glob("final_*.parquet"):
            d = p.stem.replace("final_", "")
            dates.add(d)
    return sorted(dates), sorted(set(bands))

In [14]:
# Discover bands and dates (same as notebook 2)
training_dates, bands_training = get_dates_and_bands(training_dir)
testing_dates, bands_testing = get_dates_and_bands(testing_dir)
class_options = sorted(set(bands_training) | set(bands_testing))

### ARIMA hyperparameters

In [15]:
# Window: number of past days used to fit ARIMA and predict next day (same idea as LSTM seq_len)
WINDOW = 14
# ARIMA order (p, d, q)
ARIMA_ORDER = (1, 0, 1)

### Load day-series and build keys

For each (hour, freq_center_ghz, threshold_dbm) we have a time series of au_pct by date. We use the last `WINDOW` days to fit ARIMA and forecast the next day's au_pct.

In [16]:
def load_date_from_train_or_test(band: str, date_yyyymmdd: str) -> Optional[pd.DataFrame]:
    """Load a single day's data from training_dir or testing_dir."""
    df = load_final_parquet(training_dir, band, date_yyyymmdd)
    if df is None:
        df = load_final_parquet(testing_dir, band, date_yyyymmdd)
    return df


def build_day_series_for_band(band: str, dates: list[str]) -> tuple[dict, list]:
    """
    Load all dates for band and build by_date dict and list of (hour, freq, thresh) keys.
    Returns (by_date, keys_list).
    """
    key_cols = ["hour", "freq_center_ghz", "threshold_dbm"]
    by_date = {}
    keys_set = set()
    for d in dates:
        df = load_date_from_train_or_test(band, d)
        if df is None or df.empty:
            continue
        by_date[d] = df[key_cols + ["au_pct"]].copy()
        for _, row in df[key_cols].drop_duplicates().iterrows():
            keys_set.add((int(row["hour"]), float(row["freq_center_ghz"]), int(row["threshold_dbm"])))
    keys_list = sorted(keys_set)
    return by_date, keys_list

In [17]:
def arima_forecast_one(series: np.ndarray, order: tuple) -> float:
    """Fit ARIMA on series and return one-step forecast. Returns np.nan on failure."""
    if len(series) < max(order[0], order[2], 1):
        return float(np.nanmean(series)) if len(series) > 0 else np.nan
    # Fill any remaining NaNs for statsmodels
    series = pd.Series(series).ffill().bfill().values
    if np.isnan(series).any() or not np.isfinite(series).all():
        return float(np.nanmean(series))
    try:
        model = ARIMA(series, order=order)
        fitted = model.fit(disp=False)
        return float(fitted.forecast(1).iloc[0])
    except Exception:
        return float(np.mean(series))

### Predict next day for a given date

For each (hour, freq, threshold) we use the last WINDOW days ending the day before the target, fit ARIMA, and forecast one step.

In [18]:
def get_ordered_dates_up_to(target_date_yyyymmdd: str) -> list[str]:
    """All dates (training + testing) up to and including the day before target."""
    from datetime import datetime, timedelta
    try:
        dt = datetime.strptime(target_date_yyyymmdd, "%Y%m%d")
        prev = (dt - timedelta(days=1)).strftime("%Y%m%d")
    except Exception:
        return []
    all_dates = sorted(set(training_dates) | set(testing_dates))
    return [d for d in all_dates if d <= prev]


def predict_one_day_arima(
    band: str,
    target_date: str,
    keys_list: list,
    by_date_cache: dict,
    window: int,
    order: tuple,
) -> Optional[pd.DataFrame]:
    """
    Predict au_pct for target_date for all (hour, freq, threshold) in keys_list.
    Uses last `window` days before target_date, fits ARIMA per key, forecasts 1 step.
    """
    ordered = get_ordered_dates_up_to(target_date)
    if len(ordered) < 1:
        return None
    use_dates = ordered[-window:] if len(ordered) >= window else ordered
    key_cols = ["hour", "freq_center_ghz", "threshold_dbm"]
    rows = []
    for (hour, freq, thresh) in keys_list:
        series = []
        for d in use_dates:
            if d not in by_date_cache:
                df = load_date_from_train_or_test(band, d)
                by_date_cache[d] = df[key_cols + ["au_pct"]].copy() if df is not None else None
            df = by_date_cache.get(d)
            if df is None or df.empty:
                series.append(np.nan)
                continue
            row = df[(df["hour"] == hour) & (df["freq_center_ghz"] == freq) & (df["threshold_dbm"] == thresh)]
            if row.empty:
                series.append(np.nan)
            else:
                series.append(float(row["au_pct"].iloc[0]))
        series = np.array(series, dtype=np.float64)
        if np.isnan(series).all():
            pred = np.nan
        else:
            pred = arima_forecast_one(series, order)
        rows.append({"hour": hour, "freq_center_ghz": freq, "threshold_dbm": thresh, "au_pct": float(pred)})
    return pd.DataFrame(rows)

In [19]:
def compute_arima_errors_one_band(
    band: str,
    keys_list: list,
    by_date_cache: dict,
    window: int,
    order: tuple,
) -> tuple[pd.DataFrame, float, float]:
    """
    For each testing date, predict and compute MAE/RMSE over the day. Also return overall testing MAE and RMSE.
    Returns (testing_per_day DataFrame, testing_MAE, testing_RMSE).
    """
    key_cols = ["hour", "freq_center_ghz", "threshold_dbm"]
    per_day_rows = []
    all_errs = []
    for date_curr in tqdm(testing_dates, desc=f"ARIMA {band}", leave=False):
        df_curr = load_final_parquet(testing_dir, band, date_curr)
        if df_curr is None or df_curr.empty:
            continue
        df_pred = predict_one_day_arima(band, date_curr, keys_list, by_date_cache, window, order)
        if df_pred is None or df_pred.empty:
            continue
        df_pred = df_pred.rename(columns={"au_pct": "au_pct_pred"})
        merge = df_curr.merge(df_pred, on=key_cols, how="inner")
        if merge.empty:
            continue
        # Drop rows where prediction is NaN for error computation
        merge = merge.dropna(subset=["au_pct_pred"])
        if merge.empty:
            continue
        err = merge["au_pct"] - merge["au_pct_pred"]
        mae = float(np.abs(err).mean())
        rmse = float(np.sqrt((err ** 2).mean()))
        per_day_rows.append({"date_yyyymmdd": date_curr, "MAE": mae, "RMSE": rmse})
        all_errs.extend(err.tolist())
    test_mae = float(np.abs(np.array(all_errs)).mean()) if all_errs else np.nan
    test_rmse = float(np.sqrt((np.array(all_errs) ** 2).mean())) if all_errs else np.nan
    per_day = pd.DataFrame(per_day_rows) if per_day_rows else pd.DataFrame(columns=["date_yyyymmdd", "MAE", "RMSE"])
    return per_day, test_mae, test_rmse

### Fit ARIMA per band and evaluate on testing (same structure as notebook 2)

In [20]:
effective_window = min(WINDOW, len(training_dates) + len(testing_dates) - 1)
if effective_window < 1:
    raise ValueError("Need at least 2 total dates (train+test) for a 1-day window")
print(f"Training dates: {len(training_dates)}, testing dates: {len(testing_dates)}, effective_window: {effective_window}")

results = {}
keys_per_band = {}

for band in tqdm(class_options, desc="ARIMA class"):
    _, keys_per_band[band] = build_day_series_for_band(band, training_dates)
    if not keys_per_band[band]:
        results[band] = {"testing_per_day": pd.DataFrame(), "testing_MAE": np.nan, "testing_RMSE": np.nan}
        continue
    cache = {}
    testing_per_day, test_mae, test_rmse = compute_arima_errors_one_band(
        band, keys_per_band[band], cache, effective_window, ARIMA_ORDER
    )
    results[band] = {
        "testing_per_day": testing_per_day,
        "testing_MAE": test_mae,
        "testing_RMSE": test_rmse,
    }

final_results = pd.DataFrame([
    {"Class": band, "MAE": results[band]["testing_MAE"], "RMSE": results[band]["testing_RMSE"]}
    for band in class_options
])
print("ARIMA next-day prediction — Final results (testing data)")
display(final_results.round(4))

Training dates: 6, testing dates: 3, effective_window: 8


ARIMA class:   0%|          | 0/6 [00:00<?, ?it/s]

ARIMA 195MHz:   0%|          | 0/3 [00:00<?, ?it/s]

ARIMA 2441MHz:   0%|          | 0/3 [00:00<?, ?it/s]

ARIMA 3765MHz:   0%|          | 0/3 [00:00<?, ?it/s]

ARIMA 539MHz:   0%|          | 0/3 [00:00<?, ?it/s]

ARIMA 5500MHz:   0%|          | 0/3 [00:00<?, ?it/s]

ARIMA 915MHz:   0%|          | 0/3 [00:00<?, ?it/s]

ARIMA next-day prediction — Final results (testing data)


,Class,MAE,RMSE
0,195MHz,1.9321,6.8690
1,2441MHz,5.5534,8.1105
2,3765MHz,6.6251,11.7501
3,539MHz,2.5007,7.6634
4,5500MHz,2.6245,5.4861
5,915MHz,4.1039,6.2121


### Per-class view: dropdown for class — testing MAE/RMSE per day

In [21]:
class_dropdown = widgets.Dropdown(
    options=class_options,
    value=class_options[0] if class_options else None,
    description="Class:",
    style={"description_width": "50px"},
)
out = widgets.Output()


def update_arima_viz(class_band):
    with out:
        clear_output(wait=True)
        if class_band not in results:
            print(f"No results for class {class_band}")
            return
        r = results[class_band]
        test_df = r["testing_per_day"]
        test_mae = r["testing_MAE"]

        if not test_df.empty:
            test_display = test_df.copy()
            test_display["date"] = test_display["date_yyyymmdd"].str[:4] + "-" + test_display["date_yyyymmdd"].str[4:6] + "-" + test_display["date_yyyymmdd"].str[6:8]
            print(f"ARIMA next-day prediction — Class: {class_band} (testing only)")
            display(test_display[["date", "MAE", "RMSE"]].round(4))
        print(f"Testing MAE (all testing days): {test_mae:.4f}")

        fig = go.Figure()
        if not test_df.empty:
            test_dates_dash = [f"{d[:4]}-{d[4:6]}-{d[6:8]}" for d in test_df["date_yyyymmdd"]]
            fig.add_trace(go.Bar(x=test_dates_dash, y=test_df["MAE"], name="MAE (per day)", marker_color="seagreen"))
        fig.update_layout(
            title=f"ARIMA next-day MAE by date — {class_band}",
            xaxis_title="Date",
            yaxis_title="MAE",
            height=400,
        )
        fig.show()


widgets.interactive_output(update_arima_viz, {"class_band": class_dropdown})
display(widgets.HBox([class_dropdown]), out)
update_arima_viz(class_dropdown.value)

Output()

### Predicted vs Actual heatmaps

Pick a **class** and **testing date**. Left: actual AU (%) for that day. Right: ARIMA prediction (same day). Same colorscale for comparison.